# Fine-tune OMI — khởi tạo từ backbone pretrain (notebook 14)

**Dự án ACS-ECG-AI (Vinmec).** Notebook 16 — fine-tune nhánh **OMI** trên ACS-ECG 2026,
khởi tạo backbone từ checkpoint pretrain (PTB-XL supervised + MIMIC-IV-ECG self-supervised) thay vì
random init, theo đúng thiết kế đã bàn (Phương án A).

**Trạng thái: đã nối đầy đủ với dữ liệu thật, chỉ cần notebook 14 chạy xong (`RUN_MODE="full"`)
rồi đổi `RUN_MODE = "full"` ở đây và chạy toàn bộ notebook.** Bước 2 đọc thẳng cùng nguồn dữ liệu
và lặp lại đúng cách chia POOL/TEST + K-Fold của `omi_pipeline_optimized_v3.ipynb` (cùng `SEED`)
— nên TEST và fold assignment ở đây khớp chính xác với baseline from-scratch.

**Quy trình (khớp nguyên tắc GPU-cost-tiered đã dùng cho ECGFounder ở Phần I):**
1. Load backbone checkpoint + manifest từ notebook 14, xác nhận config khớp (FS, SIGNAL_LEN, preprocess).
2. **Screening 1-fold** qua 6 mức freeze/unfreeze (rẻ) → chọn mức thắng theo AUPRC.
3. **Full K-fold** (5-fold) chỉ với mức freeze/unfreeze thắng → sinh OOF cho toàn POOL.
4. Khoá threshold trên POOL-OOF (Youden's J tối đa hoá).
5. Train FINAL trên toàn POOL, đánh giá TEST giữ riêng, so sánh bootstrap ΔAUPRC với baseline
   from-scratch hiện có trong `Bao_cao_nghien_cuu.docx`.

**Còn lại cần tay (không chặn chạy `RUN_MODE="full"`):** Bước 10 — `baseline_y_prob` để so sánh
bootstrap. Champion của baseline luôn là một ensemble (trọng số fit lúc chạy, không lưu ra file),
nên không thể tự động tái tạo chính xác — cần điền thủ công xác suất TEST của baseline (đọc từ
`omi_test_predictions_omi_optimized_v3.npz`, xem hướng dẫn ở Bước 10).

**Nguyên tắc bắt buộc:**
- Threshold khoá trên POOL-OOF, áp KHÔNG ĐỔI lên TEST — không bao giờ refit trên TEST.
- Patient-level split, K-Fold **giống hệt** `omi_pipeline_optimized_v3.ipynb` (cùng SEED) để so
  sánh công bằng với baseline from-scratch.
- TEST chỉ được chạm ở Bước 9 (đánh giá cuối cùng), không dùng cho bất kỳ lựa chọn nào trước đó.

In [ ]:
import os, sys, json, hashlib, random, time, re, shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except ImportError:
    raise SystemExit("Chưa có torch — chạy trên Colab GPU runtime (Runtime > Change runtime type > GPU).")

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_curve, confusion_matrix, average_precision_score, roc_auc_score

from google.colab import drive
drive.mount('/content/drive')

# ============================== CONFIG ==============================
RUN_MODE = "full"  # "debug" | "full"

SEED = 42
FS = 500
SIGNAL_LEN = 5000
NUM_LEADS = 12
N_FOLDS = 5
DEBUG_LIMIT = 200  # số record dùng khi RUN_MODE=debug -- pipeline chạy THẬT, không giả lập

TASK = "omi"                 # "stemi" | "omi"
TARGET_COL = "OMI"                   # cột nhãn thật trong CSV/train.csv -- khớp omi_pipeline_optimized_v3.ipynb mục 5
PATIENT_COL = "Patient_id"           # cột bệnh nhân thật -- khớp COL_PATIENT của omi_pipeline_optimized_v3.ipynb
EXISTING_FOLD_COL = "fold"           # gán trong load_pool_and_test() (Bước 2) -- tái lập ĐÚNG fold
                                       # StratifiedKFold patient-level của omi_pipeline_optimized_v3.ipynb
                                       # mục 8 (cùng SEED) để so sánh công bằng với baseline from-scratch

THRESHOLD_POLICY = "youden"   # "sensitivity_floor" | "youden"
SENSITIVITY_FLOOR = 0.91                   # chỉ dùng khi THRESHOLD_POLICY="sensitivity_floor"

RUN_TAG_SUFFIX = "_tier1"   # đổi tên checkpoint so với Tier 0 -- BẮT BUỘC train lại từ đầu với
                              # Focal Loss + augmentation + warmup/schedule mới, không âm thầm
                              # resume/dùng lại checkpoint Tier 0 (huấn luyện dưới cấu hình khác).

# --- Tier 2: kiến trúc pretrain thứ 2 (InceptionTime1D) để làm vườn ensemble (Bước 7/9) ---
# resnet1d dùng lại checkpoint Tier 1 (_tier1, đã train). inceptiontime1d train MỚI hoàn toàn,
# luôn full fine-tune ngay (bỏ qua screening freeze/unfreeze -- cả 2 lần chạy Tier 1 trước đều
# chọn mức gần/đúng full fine-tune, nên bỏ qua bước này cho kiến trúc 2 là hợp lý và đỡ tốn GPU).
BACKBONE_ARCH_LIST = ["resnet1d", "inceptiontime1d"]
RUN_TAG_SUFFIX_BY_ARCH = {"resnet1d": RUN_TAG_SUFFIX, "inceptiontime1d": "_tier2ic"}

# --- Tier 1: augmentation khi train (verbatim stemi_pipeline_optimized_v3.ipynb mục 9) ---
# Áp thẳng trên tín hiệu đã tiền xử lý (mV, KHÔNG z-score) -- backbone pretrain ở notebook 14
# cũng không z-score, giữ nguyên phân phối đầu vào mà backbone đã học.
AUG_ENABLE = True
AUG_MAX_SHIFT = 20            # dịch thời gian tối đa (mẫu, ~40ms @ 500Hz)
AUG_NOISE_STD = 0.02          # nhiễu Gaussian biên độ thấp (mV)
AUG_GAIN_JITTER = 0.05        # hệ số gain ngẫu nhiên ±5%/chuyển đạo
AUG_OP_PROB = 0.5             # xác suất áp mỗi phép augment gốc (3 phép đầu)
AUG_POWERLINE_PROB = 0.3      # xác suất chồng nhiễu điện lưới
AUG_POWERLINE_HZ = 50.0
AUG_POWERLINE_STD = 0.03      # biên độ nhiễu điện lưới (mV)
AUG_LEAD_DROPOUT_PROB = 0.03  # xác suất một chuyển đạo bị thay bằng nhiễu thấp

# --- Tier 1: Focal Loss + label smoothing (verbatim mục 12) -- bù mất cân bằng lớp dương ---
FOCAL_GAMMA = 2.0
LABEL_SMOOTH_EPS = 0.02

# --- Tier 1: warmup + ReduceLROnPlateau + early stopping (verbatim mục 2/13) ---
WARMUP_EPOCHS = 3
LR_PATIENCE = 5
EARLY_STOP_PATIENCE = 10

# --- Nguồn dữ liệu ACS-ECG 2026 thật (KHÁC Drive project của backbone pretrain ở dưới) ---
# Cùng chuẩn với omi_pipeline_optimized_v3.ipynb mục 3-5: train.csv trong CSV/, waveform
# .dat/.hea trong row_data/ (hoặc raw_data/), cùng danh sách bản ghi lỗi đọc thật ở nguồn.
DATA_DRIVE_PROJECT = Path("/content/drive/MyDrive/ACS-ECG-AI")
DATA_DRIVE_ZIP = DATA_DRIVE_PROJECT / "datasets.zip"
DATA_ROOT = Path("/content/acs_ecg_datasets")
KNOWN_BROKEN_RECORDS = ["03228", "14262"]   # verbatim từ omi_pipeline_optimized_v3.ipynb mục 2
TEST_SIZE = 0.15                            # verbatim mục 2 -- 15% bệnh nhân giữ riêng làm TEST

DRIVE_ROOT = Path("/content/drive/MyDrive/ACS-ECG-AI_pretrain_finetune")
PRETRAIN_MANIFEST_DIR = DRIVE_ROOT / "manifests"
PRETRAIN_MODELS_DIR = DRIVE_ROOT / "models" / "backbone_checkpoints"

FINETUNE_MODELS_DIR = DRIVE_ROOT / "models" / "omi_finetune"
FINETUNE_LOGS_DIR = DRIVE_ROOT / "logs" / "omi_finetune"
FINETUNE_OOF_DIR = DRIVE_ROOT / "manifests" / "omi_finetune_oof"
LOCAL_SIGNAL_CACHE_DIR = Path("/content/omi_finetune_work/cache")
DRIVE_SIGNAL_CACHE_DIR = DRIVE_ROOT / "data" / "omi_signal_cache"
for d in [FINETUNE_MODELS_DIR, FINETUNE_LOGS_DIR, FINETUNE_OOF_DIR,
          LOCAL_SIGNAL_CACHE_DIR, DRIVE_SIGNAL_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"TASK={TASK} | RUN_MODE={RUN_MODE} | device={DEVICE}")


## Bước 1 — Load backbone checkpoint từ notebook 14, xác nhận config khớp

Không chỉ load trọng số mù quáng — kiểm tra `run_manifest.json` để chắc `FS`, `SIGNAL_LEN`,
`NUM_LEADS`, thông số filter/winsorize đã dùng lúc pretrain khớp với pipeline fine-tune ở đây,
tránh lỗi âm thầm (ví dụ pretrain dùng highpass khác, tiền xử lý lệch nhau).

In [ ]:
def find_backbone_manifest(arch, manifest_dir=PRETRAIN_MANIFEST_DIR):
    candidates = sorted(manifest_dir.glob(f"{arch}_real_pretrain_run_manifest.json"),
                        key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy {arch}_real_pretrain_run_manifest.json trong {manifest_dir} -- chạy "
            f"notebook 14 với BACKBONE_ARCH=\"{arch}\" (RUN_MODE=\"full\") trước khi chạy notebook này."
        )
    return candidates[-1]

BACKBONES = {}
for _arch in BACKBONE_ARCH_LIST:
    _arch_manifest_path = find_backbone_manifest(_arch)
    _arch_manifest = json.loads(_arch_manifest_path.read_text())
    print(f"[{_arch}] Dùng backbone: {_arch_manifest_path.name}")
    _arch_cfg = _arch_manifest["config"]
    assert _arch_cfg["FS"] == FS, f"[{_arch}] FS lệch: pretrain={_arch_cfg['FS']} vs finetune={FS}"
    assert _arch_cfg["SIGNAL_LEN"] == SIGNAL_LEN, f"[{_arch}] SIGNAL_LEN lệch giữa pretrain và finetune"
    assert _arch_cfg["NUM_LEADS"] == NUM_LEADS, f"[{_arch}] NUM_LEADS lệch giữa pretrain và finetune"
    _arch_ckpt_path = Path(_arch_manifest["backbone_checkpoint"])
    _arch_sha256 = hashlib.sha256(_arch_ckpt_path.read_bytes()).hexdigest()
    assert _arch_sha256 == _arch_manifest["sha256"], (
        f"[{_arch}] Checksum backbone checkpoint không khớp manifest -- file có thể đã bị sửa/hỏng, "
        f"chạy lại notebook 14 hoặc kiểm tra lại đường dẫn."
    )
    BACKBONES[_arch] = {"manifest_path": _arch_manifest_path, "manifest": _arch_manifest,
                        "ckpt_path": _arch_ckpt_path, "sha256": _arch_sha256}

# Biến số ít cho kiến trúc chính (resnet1d) -- Tier 0/1 và export_run_manifest ở cuối dùng lại.
_manifest_path = BACKBONES["resnet1d"]["manifest_path"]
_manifest = BACKBONES["resnet1d"]["manifest"]
_pretrain_cfg = _manifest["config"]
BP_LOW = _pretrain_cfg.get("BP_LOW", 0.05)
BP_HIGH = _pretrain_cfg.get("BP_HIGH", 40.0)
BP_ORDER = _pretrain_cfg.get("BP_ORDER", 3)
WINSORIZE_MV = _pretrain_cfg.get("WINSORIZE_MV", 6.0)
print(f"\nTiền xử lý kế thừa từ pretrain: BP=[{BP_LOW},{BP_HIGH}]Hz order={BP_ORDER}, "
      f"winsorize=±{WINSORIZE_MV}mV")

BACKBONE_CKPT_PATH = BACKBONES["resnet1d"]["ckpt_path"]
_expected_sha256 = BACKBONES["resnet1d"]["sha256"]
print(f"\nBackbone checkpoint: {BACKBONE_CKPT_PATH} (SHA-256 khớp manifest)")


## Bước 2 — Load POOL/TEST thật của ACS-ECG 2026

`load_pool_and_test()` đọc thẳng cùng nguồn dữ liệu và lặp lại **đúng** logic chia
POOL/TEST + K-Fold của `omi_pipeline_optimized_v3.ipynb` (mục 3, 5, 7, 8), cùng `SEED` —
nên tập TEST và fold assignment ở đây **khớp chính xác** với baseline from-scratch, cho phép so
sánh bootstrap công bằng ở Bước 10:

- Giải nén `datasets.zip` từ `ACS-ECG-AI/` trên Drive (project dữ liệu gốc — khác
  `ACS-ECG-AI_pretrain_finetune/` chứa backbone pretrain).
- Đọc `CSV/train.csv`, nhãn lấy trực tiếp từ cột `OMI`, loại 2 bản ghi lỗi đọc thật ở nguồn
  (`KNOWN_BROKEN_RECORDS`, verbatim từ pipeline gốc).
- Tiền xử lý (bandpass + winsorize) dùng đúng `BP_LOW/BP_HIGH/BP_ORDER/WINSORIZE_MV` đã xác nhận
  khớp với pretrain ở Bước 1, cache ra 1 file `.npy` memmap resumable (cùng nguyên tắc notebook 14
  Bước 4) — POOL và TEST chỉ là 2 view khác nhau trên cùng một cache, không tách vật lý.
- Chia POOL/TEST theo bệnh nhân (`train_test_split`, `test_size=TEST_SIZE`, `random_state=SEED+200`)
  và K-Fold patient-level (`StratifiedKFold`, `random_state=SEED`) — **cùng tham số** mục 7-8 của
  pipeline gốc nên tái lập đúng tập bệnh nhân TEST và fold assignment.

In [3]:
import importlib.util
import subprocess
import zipfile

if importlib.util.find_spec("wfdb") is None:
    print("Đang cài wfdb ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
import wfdb

from scipy.signal import butter, filtfilt
from sklearn.model_selection import StratifiedKFold, train_test_split


def find_dir(root: Path, names):
    wanted = {n.lower() for n in names}
    for c in sorted(root.iterdir()):
        if c.is_dir() and c.name.lower() in wanted:
            return c
    for c in root.rglob("*"):
        if c.is_dir() and c.name.lower() in wanted:
            return c
    return None


def stage_acs_ecg_data(data_root=DATA_ROOT, drive_zip=DATA_DRIVE_ZIP):
    """Giải nén datasets.zip của ACS-ECG-AI về đĩa local Colab -- CÙNG file zip mà
    omi_pipeline_optimized_v3.ipynb dùng (mục 3), khác Drive project riêng của backbone
    pretrain (notebook 14 dùng ACS-ECG-AI_pretrain_finetune/)."""
    marker = data_root / ".staged_ok"
    if marker.exists():
        real = Path(marker.read_text().strip())
        print(f"Dữ liệu ACS-ECG đã sẵn sàng: {real}")
        return real
    if not drive_zip.exists():
        raise FileNotFoundError(
            f"Không thấy {drive_zip} -- kiểm tra thư mục ACS-ECG-AI trong MyDrive "
            f"(cùng thư mục omi_pipeline_optimized_v3.ipynb đang dùng)."
        )
    local_zip = Path("/content/_acs_ecg_dataset.zip")
    size = drive_zip.stat().st_size
    if not (local_zip.exists() and local_zip.stat().st_size == size):
        print(f"Copy zip {size / 1024 ** 3:.2f} GB từ Drive ...")
        t0 = time.time()
        shutil.copy2(drive_zip, local_zip)
        print(f"  {time.time() - t0:.0f}s")
    extract_to = Path("/content/_acs_ecg_extract")
    if extract_to.exists():
        shutil.rmtree(extract_to)
    print("Giải nén ...")
    t0 = time.time()
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(extract_to)
    print(f"  {time.time() - t0:.0f}s")

    real = extract_to
    if find_dir(extract_to, ["csv"]) is None:
        for cand in sorted(p for p in extract_to.rglob("*") if p.is_dir()):
            if find_dir(cand, ["csv"]) is not None:
                real = cand
                break
    data_root.mkdir(parents=True, exist_ok=True)
    marker.write_text(str(real))
    return real


def load_signal(stem: str, raw_dir: Path) -> np.ndarray:
    """Verbatim load_signal() của omi_pipeline_optimized_v3.ipynb mục 6 (đọc đúng số mẫu
    thực có khi .dat ngắn hơn header khai báo, thay vì để wfdb raise ValueError)."""
    path = str(raw_dir / stem)
    try:
        rec = wfdb.rdrecord(path)
    except ValueError:
        n_sig = int((raw_dir / f"{stem}.hea").read_text().splitlines()[0].split()[1])
        n = (raw_dir / f"{stem}.dat").stat().st_size // (n_sig * 2)
        rec = wfdb.rdrecord(path, sampto=n)
    return np.asarray(rec.p_signal, dtype=np.float32).T


_B, _A = butter(BP_ORDER, [BP_LOW / (FS / 2), BP_HIGH / (FS / 2)], btype="band")


def preprocess_signal(sig: np.ndarray) -> np.ndarray:
    """Verbatim preprocess() của omi_pipeline_optimized_v3.ipynb mục 6 -- BP_LOW/BP_HIGH/
    BP_ORDER/WINSORIZE_MV kế thừa từ manifest pretrain ở Bước 1 (đã xác nhận khớp)."""
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    if sig.shape[1] < SIGNAL_LEN:
        sig = np.pad(sig, ((0, 0), (0, SIGNAL_LEN - sig.shape[1])))
    filtered = filtfilt(_B, _A, sig[:, :SIGNAL_LEN], axis=1)
    winsorized = np.clip(filtered, -WINSORIZE_MV, WINSORIZE_MV)
    return np.ascontiguousarray(winsorized, dtype=np.float32)


def _cache_complete(meta_p, npy_p, n_expected):
    if not (meta_p.exists() and npy_p.exists()):
        return False
    try:
        return json.loads(meta_p.read_text()).get("n_done", 0) >= n_expected
    except (OSError, ValueError):
        return False


def build_signal_cache(record_ids, raw_dir, tag, checkpoint_every=2000):
    """Cache resumable 1 file .npy lớn (memmap), build local trước rồi backup định kỳ + cuối
    cùng lên Drive -- cùng nguyên tắc notebook 14 Bước 4 / pipeline gốc mục 6."""
    n = len(record_ids)
    npy_p = LOCAL_SIGNAL_CACHE_DIR / f"{tag}.npy"
    meta_p = LOCAL_SIGNAL_CACHE_DIR / f"{tag}.meta.json"
    d_npy_p = DRIVE_SIGNAL_CACHE_DIR / f"{tag}.npy"
    d_meta_p = DRIVE_SIGNAL_CACHE_DIR / f"{tag}.meta.json"

    if not _cache_complete(meta_p, npy_p, n) and _cache_complete(d_meta_p, d_npy_p, n):
        print(f"[{tag}] Cache đầy đủ trên Drive -- copy về local ...")
        shutil.copy2(d_npy_p, npy_p)
        shutil.copy2(d_meta_p, meta_p)

    if _cache_complete(meta_p, npy_p, n):
        print(f"[{tag}] Cache đã đầy đủ ({n:,} bản ghi), dùng lại.")
        return np.load(npy_p, mmap_mode="r")

    meta = json.loads(meta_p.read_text()) if meta_p.exists() else {}
    start = int(meta.get("n_done", 0)) if npy_p.exists() else 0
    if start:
        arr = np.lib.format.open_memmap(npy_p, mode="r+")
        print(f"[{tag}] Build tiếp từ {start:,}/{n:,}")
    else:
        arr = np.lib.format.open_memmap(npy_p, mode="w+", dtype=np.float16,
                                        shape=(n, NUM_LEADS, SIGNAL_LEN))
        print(f"[{tag}] Build cache mới {n:,} bản ghi "
              f"(~{n * NUM_LEADS * SIGNAL_LEN * 2 / 1024 ** 3:.2f} GB)")

    t0 = time.time()
    for i in range(start, n):
        arr[i] = preprocess_signal(load_signal(record_ids[i], raw_dir)).astype(np.float16)
        done = i + 1
        if done % 500 == 0 or done == n:
            el = max(time.time() - t0, 1e-6)
            print(f"  [{tag}] {done:>7,}/{n:,}  {(done - start) / el:5.1f} rec/s")
        if done % checkpoint_every == 0 or done == n:
            arr.flush()
            meta_p.write_text(json.dumps({"n_done": done}))
            shutil.copy2(npy_p, d_npy_p)
            shutil.copy2(meta_p, d_meta_p)
    del arr
    print(f"[{tag}] Xong trong {time.time() - t0:.0f}s, đã lưu Drive: {d_npy_p}")
    return np.load(npy_p, mmap_mode="r")


def load_pool_and_test():
    """Đọc thật POOL/TEST của ACS-ECG 2026 -- CÙNG nguồn dữ liệu, CÙNG cách chia bệnh nhân/fold
    (cùng SEED) với omi_pipeline_optimized_v3.ipynb (mục 3-8), để TEST và fold assignment
    KHỚP CHÍNH XÁC với baseline from-scratch dùng cho so sánh bootstrap ở Bước 10.

    Trả về: pool_df, test_df (cột record_id/{PATIENT_COL}/{TARGET_COL}/_cache_idx, pool_df có
    thêm cột {EXISTING_FOLD_COL}), pool_cache, test_cache -- CÙNG một mảng memmap, index theo
    _cache_idx (POOL/TEST chỉ là 2 view khác nhau trên cùng cache, không tách vật lý).
    """
    data_root = stage_acs_ecg_data()
    raw_dir = find_dir(data_root, ["row_data", "raw_data"])
    csv_dir = find_dir(data_root, ["csv"])
    assert raw_dir and csv_dir, f"Không thấy row_data/raw_data hoặc CSV trong {data_root}"

    df_raw = pd.read_csv(csv_dir / "train.csv")
    df_raw["record_id"] = df_raw["ecg_row_record"].astype(str).str.replace(".dat", "", regex=False)
    df_raw[TARGET_COL] = df_raw[TARGET_COL].astype(int)

    _broken = df_raw["record_id"].isin(KNOWN_BROKEN_RECORDS)
    if _broken.any():
        print(f"Loại {int(_broken.sum())} bản ghi lỗi đọc thật ở nguồn: "
              f"{df_raw.loc[_broken, 'record_id'].tolist()}")
        df_raw = df_raw[~_broken].reset_index(drop=True)

    df_all = df_raw[["record_id", PATIENT_COL, TARGET_COL]].copy()

    if RUN_MODE == "debug" and len(df_all) > DEBUG_LIMIT:
        df_all, _ = train_test_split(df_all, train_size=DEBUG_LIMIT,
                                     stratify=df_all[TARGET_COL], random_state=SEED)
    df_all = df_all.reset_index(drop=True)

    n_pos = int(df_all[TARGET_COL].sum())
    print(f"{TASK.upper()}: {len(df_all):,} bản ghi | {n_pos:,} dương ({n_pos / len(df_all):.2%})")
    assert 0 < n_pos < len(df_all), "Tập chỉ có một lớp -- không train được."

    preprocess_tag = f"hp{BP_LOW}_{BP_HIGH}hz_o{BP_ORDER}_ws{WINSORIZE_MV}"
    full_cache = build_signal_cache(df_all["record_id"].tolist(), raw_dir,
                                    tag=f"{TASK}_{preprocess_tag}_n{len(df_all)}")
    df_all["_cache_idx"] = np.arange(len(df_all))

    # --- Chia POOL/TEST theo bệnh nhân -- ĐÚNG omi_pipeline_optimized_v3.ipynb mục 7,
    # cùng SEED+200 -> tái lập CHÍNH XÁC cùng bệnh nhân TEST của baseline from-scratch ---
    pat_all = df_all.groupby(PATIENT_COL)[TARGET_COL].max().reset_index()
    pat_pool, pat_test = train_test_split(pat_all, test_size=TEST_SIZE,
                                          stratify=pat_all[TARGET_COL], random_state=SEED + 200)
    test_patients = set(pat_test[PATIENT_COL])
    is_test = df_all[PATIENT_COL].isin(test_patients).to_numpy()
    pool_df = df_all[~is_test].reset_index(drop=True)
    test_df = df_all[is_test].reset_index(drop=True)
    assert not (set(pool_df[PATIENT_COL]) & test_patients), "Rò rỉ bệnh nhân TEST vào POOL"

    # --- K-fold patient-level trong POOL -- ĐÚNG mục 8, cùng SEED -> tái lập CHÍNH XÁC cùng
    # fold assignment của baseline (miễn POOL patient set giống nhau, đã đảm bảo ở bước trên) ---
    pat_tbl = pool_df.groupby(PATIENT_COL)[TARGET_COL].max().reset_index()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_of_patient = {}
    for k, (_, va_pos) in enumerate(skf.split(pat_tbl, pat_tbl[TARGET_COL])):
        for pid in pat_tbl.iloc[va_pos][PATIENT_COL]:
            fold_of_patient[pid] = k
    pool_df[EXISTING_FOLD_COL] = pool_df[PATIENT_COL].map(fold_of_patient)
    assert pool_df[EXISTING_FOLD_COL].notna().all(), "Có bệnh nhân POOL chưa được gán fold"
    pool_df[EXISTING_FOLD_COL] = pool_df[EXISTING_FOLD_COL].astype(int)

    print(f"POOL: {len(pool_df):,} bản ghi / {pool_df[PATIENT_COL].nunique():,} bệnh nhân | "
          f"TEST: {len(test_df):,} bản ghi / {test_df[PATIENT_COL].nunique():,} bệnh nhân")
    print(f"Tỷ lệ dương POOL: {pool_df[TARGET_COL].mean():.2%} | TEST: {test_df[TARGET_COL].mean():.2%}")
    return pool_df, test_df, full_cache, full_cache


pool_df, test_df, pool_cache, test_cache = load_pool_and_test()

Đang cài wfdb ...
Copy zip 1.33 GB từ Drive ...
  17s
Giải nén ...
  27s
Loại 2 bản ghi lỗi đọc thật ở nguồn: ['14262', '03228']
OMI: 17,958 bản ghi | 1,151 dương (6.41%)
[omi_hp0.05_40.0hz_o3_ws6.0_n17958] Cache đầy đủ trên Drive -- copy về local ...
[omi_hp0.05_40.0hz_o3_ws6.0_n17958] Cache đã đầy đủ (17,958 bản ghi), dùng lại.
POOL: 15,256 bản ghi / 14,463 bệnh nhân | TEST: 2,702 bản ghi / 2,553 bệnh nhân
Tỷ lệ dương POOL: 6.40% | TEST: 6.44%


## Bước 3 — K-fold patient-level (tái dùng fold có sẵn nếu POOL đã gán từ trước)

In [4]:
def make_or_reuse_folds(pool_df, n_folds=N_FOLDS, seed=SEED,
                         patient_col=None, existing_fold_col=EXISTING_FOLD_COL, target_col=None):
    patient_col = patient_col or PATIENT_COL
    target_col = target_col or TARGET_COL
    if existing_fold_col in pool_df.columns:
        print(f"Dùng lại cột fold có sẵn '{existing_fold_col}' -- khớp fold của baseline from-scratch.")
        return pool_df[existing_fold_col].to_numpy()
    print(f"Không thấy cột '{existing_fold_col}' -- tự chia {n_folds}-fold patient-level mới "
          f"(StratifiedGroupKFold). LƯU Ý: fold này sẽ KHÁC baseline nếu baseline dùng fold khác --"
          f" so sánh bootstrap ở Bước 7 vẫn hợp lệ (cùng TEST) nhưng OOF không trực tiếp so được.")
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold_assign = np.full(len(pool_df), -1, dtype=int)
    for fold_id, (_, val_idx) in enumerate(
            sgkf.split(pool_df, pool_df[target_col], groups=pool_df[patient_col])):
        fold_assign[val_idx] = fold_id
    return fold_assign

pool_df["_fold"] = make_or_reuse_folds(pool_df)

# kiểm tra không rò rỉ bệnh nhân giữa các fold
_leak = (pool_df.groupby(PATIENT_COL)["_fold"].nunique() > 1)
assert not _leak.any(), f"Rò rỉ bệnh nhân giữa các fold: {_leak[_leak].index.tolist()[:5]}..."
print(pool_df["_fold"].value_counts().sort_index())

Dùng lại cột fold có sẵn 'fold' -- khớp fold của baseline from-scratch.
_fold
0    3049
1    3050
2    3058
3    3039
4    3060
Name: count, dtype: int64


## Bước 4 — Backbone + head + freeze/unfreeze (6 mức, chỉ resnet1d)

Backbone **phải cùng kiến trúc/tham số** với notebook 14 để `load_state_dict()` khớp.
**Tier 2:** thêm `InceptionTime1DBackbone` (kiến trúc thứ 2), chọn qua `BACKBONE_CLASSES[arch]`. 6 mức freeze/unfreeze (`apply_freeze_level`) chỉ áp dụng cho resnet1d (có 4 stage blocks rõ ràng); inceptiontime1d dùng `unfreeze_all_backbone()` -- full fine-tune ngay, không qua screening (xem lý do ở Bước 5/7).


In [ ]:
class ResidualBlock1D(nn.Module):
    """Verbatim từ stemi_pipeline_optimized_v3.ipynb mục 11 -- PHẢI khớp notebook 14."""
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        return self.relu(self.bn2(self.conv2(out)) + idt)

class ResNet1DBackbone(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256), in_ch=NUM_LEADS):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_ch, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(ResidualBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.out_dim = c_in

    def forward(self, x):
        x = self.blocks(self.stem(x))
        return self.pool(x).squeeze(-1)


class InceptionModule(nn.Module):
    """Verbatim từ stemi_pipeline_optimized_v3.ipynb mục 11 -- PHẢI khớp notebook 14."""
    def __init__(self, c_in, n_filters=32, kernels=(39, 19, 9), bottleneck=32):
        super().__init__()
        self.bottleneck = nn.Conv1d(c_in, bottleneck, 1, bias=False)
        self.convs = nn.ModuleList(
            [nn.Conv1d(bottleneck, n_filters, k, padding=k // 2, bias=False) for k in kernels])
        self.pool_conv = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1),
                                       nn.Conv1d(c_in, n_filters, 1, bias=False))
        self.bn = nn.BatchNorm1d(n_filters * (len(kernels) + 1))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        b = self.bottleneck(x)
        return self.relu(self.bn(torch.cat([c(b) for c in self.convs] + [self.pool_conv(x)], 1)))


class InceptionTime1DBackbone(nn.Module):
    """Verbatim từ stemi_pipeline_optimized_v3.ipynb mục 11 (InceptionTime1D) -- kiến trúc thứ 2
    của Tier 2, PHẢI khớp notebook 14."""
    def __init__(self, n_filters=32, depth=6, in_ch=NUM_LEADS):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_ch, 32, 15, 4, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True))
        c_out = n_filters * 4
        self.blocks = nn.ModuleList()
        self.shortcuts = nn.ModuleList()
        self.pools = nn.ModuleList()
        c_in = 32
        res_c = 32
        for d in range(depth):
            self.blocks.append(InceptionModule(c_in, n_filters))
            if d % 3 == 2:
                self.shortcuts.append(nn.Sequential(nn.Conv1d(res_c, c_out, 1, bias=False),
                                                    nn.BatchNorm1d(c_out)))
                self.pools.append(nn.MaxPool1d(4))
                res_c = c_out
            else:
                self.shortcuts.append(None)
                self.pools.append(None)
            c_in = c_out
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.out_dim = c_out

    def forward(self, x):
        x = self.stem(x)
        res = x
        for blk, short, pool in zip(self.blocks, self.shortcuts, self.pools):
            x = blk(x)
            if short is not None:
                x = self.relu(x + short(res))
                x = pool(x)
                res = x
        return self.pool(x).squeeze(-1)


BACKBONE_CLASSES = {"resnet1d": ResNet1DBackbone, "inceptiontime1d": InceptionTime1DBackbone}

class BinaryHead(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)
    def forward(self, x):
        return self.fc(x).squeeze(-1)

class FinetuneModel(nn.Module):
    def __init__(self, arch="resnet1d"):
        super().__init__()
        self.arch = arch
        self.backbone = BACKBONE_CLASSES[arch]()
        self.head = BinaryHead(self.backbone.out_dim)
    def forward(self, x):
        return self.head(self.backbone(x))

def load_pretrained_backbone(model, arch="resnet1d"):
    # weights_only=False: checkpoint tự tạo (notebook 14), đáng tin cậy -- mặc định
    # weights_only=True của torch>=2.6 chặn cả state_dict thuần tensor một cách không cần thiết.
    ckpt_path = BACKBONES[arch]["ckpt_path"]
    state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.backbone.load_state_dict(state, strict=True)
    return model

FREEZE_LEVELS = {
    0: "freeze_all",         # chỉ train head (linear probe)
    1: "unfreeze_block4",    # + block cuối
    2: "unfreeze_block34",   # + 2 block cuối
    3: "unfreeze_block234",  # + 3 block cuối
    4: "unfreeze_all_blocks",# toàn bộ blocks, giữ nguyên stem
    5: "full_finetune",      # toàn bộ backbone + stem
}

def apply_freeze_level(model, level):
    """Chỉ dùng cho resnet1d -- kiến trúc có 4 stage blocks rõ ràng để chia mức đóng/mở băng.
    InceptionTime1D (Tier 2) dùng unfreeze_all_backbone() thay vì hàm này (xem Bước 5)."""
    assert level in FREEZE_LEVELS, f"level phải trong {list(FREEZE_LEVELS)}"
    for p in model.backbone.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True
    n_blocks = len(model.backbone.blocks)
    if level >= 1:
        for p in model.backbone.blocks[n_blocks - 1].parameters(): p.requires_grad = True
    if level >= 2:
        for p in model.backbone.blocks[n_blocks - 2].parameters(): p.requires_grad = True
    if level >= 3:
        for p in model.backbone.blocks[n_blocks - 3].parameters(): p.requires_grad = True
    if level >= 4:
        for blk in model.backbone.blocks:
            for p in blk.parameters(): p.requires_grad = True
    if level >= 5:
        for p in model.backbone.stem.parameters(): p.requires_grad = True
    return FREEZE_LEVELS[level]

def unfreeze_all_backbone(model):
    """Dùng cho kiến trúc Tier 2 (inceptiontime1d) -- bỏ qua giai đoạn screening freeze/unfreeze
    vì cả 2 lần chạy Tier 1 (STEMI, OMI) đều chọn mức gần/đúng full fine-tune, nên full fine-tune
    ngay là lựa chọn hợp lý và đỡ tốn GPU cho kiến trúc thứ 2."""
    for p in model.parameters():
        p.requires_grad = True

for _arch in BACKBONE_ARCH_LIST:
    _sanity_model = FinetuneModel(arch=_arch)
    load_pretrained_backbone(_sanity_model, arch=_arch)
    _x = torch.randn(2, NUM_LEADS, SIGNAL_LEN)
    assert _sanity_model(_x).shape == (2,)
    if _arch == "resnet1d":
        for _lvl in range(6):
            apply_freeze_level(_sanity_model, _lvl)
    else:
        unfreeze_all_backbone(_sanity_model)
    _n_trainable = sum(p.numel() for p in _sanity_model.parameters() if p.requires_grad)
    print(f"[{_arch}] Load backbone pretrain + sanity check OK. {_n_trainable:,} tham số trainable (full).")
    del _sanity_model


## Bước 5 — Dataset & training loop (dùng chung cho screening + full K-fold)

**Tier 1:** thêm 5 phép augmentation khi train (verbatim `stemi_pipeline_optimized_v3.ipynb`
mục 9: time-shift, nhiễu Gaussian, gain jitter, nhiễu điện lưới, lead-dropout), Focal Loss +
label smoothing thay BCE trơn (bù mất cân bằng lớp dương ~6-8%), warmup tuyến tính +
`ReduceLROnPlateau` + early stopping thay vì train cố định n_epochs. Checkpoint đổi sang
`RUN_TAG_SUFFIX="_tier1"` nên sẽ train lại từ đầu (từ backbone pretrain), không đụng vào
checkpoint Tier 0 đã có.

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau


def _time_shift(x: np.ndarray, max_shift: int) -> np.ndarray:
    shift = np.random.randint(-max_shift, max_shift + 1)
    if shift == 0:
        return x
    if shift > 0:
        return np.pad(x, ((0, 0), (shift, 0)), mode="edge")[:, :x.shape[1]]
    return np.pad(x, ((0, 0), (0, -shift)), mode="edge")[:, -x.shape[1]:]


def _powerline_noise(x: np.ndarray, fs: int, hz: float, std: float) -> np.ndarray:
    """Nhiễu điện lưới chồng lên MỌI chuyển đạo cùng lúc, biên độ/pha ngẫu nhiên mỗi mẫu --
    verbatim stemi_pipeline_optimized_v3.ipynb mục 9."""
    t = np.arange(x.shape[1]) / fs
    phase = np.random.uniform(0, 2 * np.pi)
    amp = np.random.uniform(0.3, 1.0) * std
    noise = (amp * np.sin(2 * np.pi * hz * t + phase)).astype(np.float32)
    return x + noise[None, :]


def _lead_dropout(x: np.ndarray) -> np.ndarray:
    """Mô phỏng một chuyển đạo bị rớt/tiếp xúc kém -- verbatim mục 9."""
    x = x.copy()
    lead = np.random.randint(0, x.shape[0])
    x[lead] = np.random.normal(0.0, 0.05, size=x.shape[1]).astype(np.float32)
    return x


def augment_ecg(x: np.ndarray) -> np.ndarray:
    """5 phép augment của stemi_pipeline_optimized_v3.ipynb mục 9."""
    if np.random.rand() < AUG_OP_PROB:
        x = _time_shift(x, AUG_MAX_SHIFT)
    if np.random.rand() < AUG_OP_PROB:
        x = x + np.random.normal(0.0, AUG_NOISE_STD, size=x.shape).astype(np.float32)
    if np.random.rand() < AUG_OP_PROB:
        gain = np.random.uniform(1 - AUG_GAIN_JITTER, 1 + AUG_GAIN_JITTER,
                                 size=(NUM_LEADS, 1)).astype(np.float32)
        x = x * gain
    if np.random.rand() < AUG_POWERLINE_PROB:
        x = _powerline_noise(x, FS, AUG_POWERLINE_HZ, AUG_POWERLINE_STD)
    if np.random.rand() < AUG_LEAD_DROPOUT_PROB:
        x = _lead_dropout(x)
    return x


class FocalLossWithSmoothing(nn.Module):
    """BCE có trọng số alpha (theo tỷ lệ lớp thật) + điều chỉnh focal gamma (tập trung ca khó)
    + label smoothing -- verbatim stemi_pipeline_optimized_v3.ipynb mục 12."""

    def __init__(self, alpha_pos: float, gamma: float = 2.0, label_smoothing: float = 0.0):
        super().__init__()
        assert 0.0 < alpha_pos < 1.0
        self.alpha_pos = float(alpha_pos)
        self.gamma = float(gamma)
        self.eps = float(label_smoothing)

    def forward(self, logits, targets):
        targets = targets.float()
        targets_smooth = targets * (1 - self.eps) + self.eps / 2 if self.eps > 0 else targets
        bce = F.binary_cross_entropy_with_logits(logits, targets_smooth, reduction="none")
        if self.gamma <= 0 and self.alpha_pos == 0.5:
            return bce.mean()
        with torch.no_grad():
            p = torch.sigmoid(logits)
            p_t = p * targets + (1 - p) * (1 - targets)
            alpha_t = self.alpha_pos * targets + (1 - self.alpha_pos) * (1 - targets)
            focal_w = alpha_t * (1 - p_t).clamp(min=1e-6, max=1.0) ** self.gamma
        return (focal_w * bce).mean()


class ECGBinaryDataset(Dataset):
    def __init__(self, df, cache_array, target_col=TARGET_COL, augment=False):
        self.df = df.reset_index(drop=True)
        self.cache = cache_array
        self.target_col = target_col
        self.augment = augment and AUG_ENABLE

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = np.asarray(self.cache[row["_cache_idx"]], dtype=np.float32)
        if self.augment:
            x = augment_ecg(x)
        y = np.float32(row[self.target_col])
        return torch.from_numpy(np.ascontiguousarray(x)), torch.tensor(y)


def save_checkpoint(model, optimizer, epoch, val_auprc, path):
    torch.save({"epoch": epoch, "model_state": model.state_dict(),
               "optimizer_state": optimizer.state_dict(), "val_auprc": val_auprc}, path)


def load_checkpoint_if_exists(model, optimizer, path):
    if path.exists():
        # weights_only=False: checkpoint tự tạo trong chính notebook này (chứa val_auprc kiểu
        # numpy scalar) -- mặc định weights_only=True của torch>=2.6 sẽ raise UnpicklingError.
        ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        print(f"Resume từ checkpoint epoch={ckpt['epoch']}, val_auprc={ckpt['val_auprc']:.4f}")
        return ckpt["epoch"] + 1
    return 0


def _apply_warmup_lr(optimizer, base_lr, epoch):
    """Warmup tuyến tính WARMUP_EPOCHS epoch đầu -- verbatim stemi_pipeline_optimized_v3.ipynb.
    Trả về True nếu vẫn đang trong giai đoạn warmup (scheduler CHƯA được can thiệp)."""
    if WARMUP_EPOCHS <= 0 or epoch >= WARMUP_EPOCHS:
        return False
    for g in optimizer.param_groups:
        g["lr"] = base_lr * (epoch + 1) / WARMUP_EPOCHS
    return epoch < WARMUP_EPOCHS - 1


def train_one_run(train_loader, val_loader, freeze_level, run_tag, n_epochs=None, arch="resnet1d"):
    """Train 1 model (1 fold hoặc screening) với 1 mức freeze/unfreeze, trả về best val AUPRC
    + đường dẫn checkpoint tốt nhất + xác suất dự đoán OOF trên val_loader (dùng checkpoint tốt
    nhất). Tier 1: Focal Loss + label smoothing (bù mất cân bằng lớp dương), warmup +
    ReduceLROnPlateau (ổn định hội tụ), early stopping (dừng đúng lúc thay vì train cố định
    n_epochs), grad clipping -- cùng bộ kỹ thuật stemi_pipeline_optimized_v3.ipynb đã dùng."""
    model = FinetuneModel(arch=arch).to(DEVICE)
    load_pretrained_backbone(model, arch=arch)
    if arch == "resnet1d":
        apply_freeze_level(model, freeze_level)
    else:
        unfreeze_all_backbone(model)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    base_lr = 3e-4
    optimizer = torch.optim.AdamW(trainable_params, lr=base_lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=LR_PATIENCE)

    alpha_pos = 1.0 - float(train_loader.dataset.df[train_loader.dataset.target_col].mean())
    criterion = FocalLossWithSmoothing(alpha_pos=alpha_pos, gamma=FOCAL_GAMMA,
                                       label_smoothing=LABEL_SMOOTH_EPS)

    ckpt_path = FINETUNE_MODELS_DIR / f"{run_tag}_latest.pt"
    best_path = FINETUNE_MODELS_DIR / f"{run_tag}_best.pt"
    start_epoch = load_checkpoint_if_exists(model, optimizer, ckpt_path)

    n_epochs = n_epochs or (2 if RUN_MODE == "debug" else 30)
    best_auprc = -1.0
    best_y_true = best_y_prob = None
    last_y_true = last_y_prob = None
    bad_epochs = 0

    for epoch in range(start_epoch, n_epochs):
        in_warmup = _apply_warmup_lr(optimizer, base_lr, epoch)
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        y_true_all, y_prob_all = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(DEVICE)
                prob = torch.sigmoid(model(x)).cpu().numpy()
                y_true_all.append(y.numpy())
                y_prob_all.append(prob)
        y_true_all = np.concatenate(y_true_all)
        y_prob_all = np.concatenate(y_prob_all)
        auprc = average_precision_score(y_true_all, y_prob_all) if y_true_all.sum() > 0 else float("nan")
        if not in_warmup:
            scheduler.step(auprc if not np.isnan(auprc) else -np.inf)

        print(f"  [{run_tag}] epoch {epoch + 1}/{n_epochs} -- val_AUPRC={auprc:.4f} "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")
        last_y_true, last_y_prob = y_true_all, y_prob_all
        save_checkpoint(model, optimizer, epoch, auprc, ckpt_path)
        if not np.isnan(auprc) and auprc > best_auprc:
            best_auprc = auprc
            bad_epochs = 0
            save_checkpoint(model, optimizer, epoch, auprc, best_path)
            best_y_true, best_y_prob = y_true_all, y_prob_all
        else:
            bad_epochs += 1
        if bad_epochs >= EARLY_STOP_PATIENCE:
            print(f"  [{run_tag}] early stop @ epoch {epoch + 1} (không cải thiện "
                  f"{EARLY_STOP_PATIENCE} epoch liên tiếp)")
            break

    if best_y_true is None:
        if start_epoch >= n_epochs and best_path.exists():
            # Checkpoint đã train đủ/early-stop từ MỘT LẦN CHẠY TRƯỚC -- vòng for phía trên
            # không chạy bước nào. best_path trên đĩa đã đúng (checkpoint của epoch tốt nhất
            # thật sự) -- TUYỆT ĐỐI không ghi đè bằng ckpt_path (epoch cuối), chỉ eval lại bằng
            # đúng trọng số best đã lưu để lấy y_true/y_prob cho OOF.
            print(f"  [{run_tag}] Đã hoàn tất từ trước -- eval lại bằng checkpoint tốt nhất đã "
                  f"lưu, không train/ghi đè thêm.")
            _best_state = torch.load(best_path, map_location=DEVICE, weights_only=False)
            model.load_state_dict(_best_state["model_state"])
            model.eval()
            y_true_all, y_prob_all = [], []
            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(DEVICE)
                    prob = torch.sigmoid(model(x)).cpu().numpy()
                    y_true_all.append(y.numpy())
                    y_prob_all.append(prob)
            best_y_true = np.concatenate(y_true_all)
            best_y_prob = np.concatenate(y_prob_all)
            best_auprc = (average_precision_score(best_y_true, best_y_prob)
                          if best_y_true.sum() > 0 else float(_best_state.get("val_auprc", float("nan"))))
        else:
            # Không epoch nào có AUPRC hợp lệ -- val fold thiếu lớp dương (dễ gặp với target hiếm như
            # STEMI/OMI ở fold nhỏ). Fallback: dùng checkpoint + dự đoán epoch cuối thay vì crash.
            print(f"  [CẢNH BÁO] {run_tag}: không epoch nào có AUPRC hợp lệ (val fold có thể thiếu lớp "
                  f"dương) -- dùng checkpoint epoch cuối làm fallback.")
            shutil.copy2(ckpt_path, best_path)
            best_y_true, best_y_prob = last_y_true, last_y_prob

    return {"best_auprc": best_auprc, "best_ckpt": best_path,
            "oof_y_true": best_y_true, "oof_y_prob": best_y_prob}


## Bước 6 — Giai đoạn A: screening 1-fold qua 6 mức freeze/unfreeze

In [7]:
BATCH_SIZE = 8 if RUN_MODE == "debug" else 64

screen_fold = 0
screen_train_df = pool_df[pool_df["_fold"] != screen_fold]
screen_val_df = pool_df[pool_df["_fold"] == screen_fold]

# pool_cache: mảng waveform đã tiền xử lý, index khớp cột "_cache_idx" -- gán ở Bước 2
# (load_pool_and_test), cùng cache vật lý dùng chung với test_cache.
screen_train_loader = DataLoader(ECGBinaryDataset(screen_train_df, pool_cache, augment=True),
                                  batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
screen_val_loader = DataLoader(ECGBinaryDataset(screen_val_df, pool_cache),
                                batch_size=BATCH_SIZE, shuffle=False)
assert len(screen_train_loader) > 0, "screen_train_loader rỗng -- tăng DEBUG_LIMIT hoặc giảm BATCH_SIZE"

screening_results = {}
for level in range(6):
    print(f"\n=== Screening mức freeze/unfreeze {level} ({FREEZE_LEVELS[level]}) ===")
    res = train_one_run(screen_train_loader, screen_val_loader, level,
                         run_tag=f"{TASK}_screen_lvl{level}{RUN_TAG_SUFFIX}", n_epochs=2 if RUN_MODE == "debug" else 8)
    screening_results[level] = res["best_auprc"]

best_level = max(screening_results, key=screening_results.get)
print("\nKết quả screening:", {FREEZE_LEVELS[k]: round(v, 4) for k, v in screening_results.items()})
print(f"Mức thắng: {best_level} ({FREEZE_LEVELS[best_level]}) -- dùng cho full K-fold ở Bước 7.")


=== Screening mức freeze/unfreeze 0 (freeze_all) ===
Resume từ checkpoint epoch=7, val_auprc=0.2541
  [omi_screen_lvl0_tier1] Đã hoàn tất từ trước -- eval lại bằng checkpoint tốt nhất đã lưu, không train/ghi đè thêm.

=== Screening mức freeze/unfreeze 1 (unfreeze_block4) ===
Resume từ checkpoint epoch=7, val_auprc=0.3294
  [omi_screen_lvl1_tier1] Đã hoàn tất từ trước -- eval lại bằng checkpoint tốt nhất đã lưu, không train/ghi đè thêm.

=== Screening mức freeze/unfreeze 2 (unfreeze_block34) ===
Resume từ checkpoint epoch=7, val_auprc=0.3473
  [omi_screen_lvl2_tier1] Đã hoàn tất từ trước -- eval lại bằng checkpoint tốt nhất đã lưu, không train/ghi đè thêm.

=== Screening mức freeze/unfreeze 3 (unfreeze_block234) ===
Resume từ checkpoint epoch=7, val_auprc=0.2915
  [omi_screen_lvl3_tier1] Đã hoàn tất từ trước -- eval lại bằng checkpoint tốt nhất đã lưu, không train/ghi đè thêm.

=== Screening mức freeze/unfreeze 4 (unfreeze_all_blocks) ===
Resume từ checkpoint epoch=7, val_auprc=0.3650


## Bước 7 — Tier 2: K-fold cho mọi kiến trúc trong `BACKBONE_ARCH_LIST` → vườn ensemble

**Cập nhật (Tier 2):** vòng lặp ngoài chạy K-fold cho TỪNG kiến trúc (`resnet1d` dùng lại run_tag `_tier1`, `inceptiontime1d` train mới run_tag `_tier2ic`, luôn full fine-tune). Mỗi kiến trúc được hiệu chuẩn Platt (`crossfit_platt`, verbatim `stemi_pipeline_optimized_v3.ipynb` mục 16) riêng. Sau đó dựng **vườn ensemble** (mean / logit-mean / stacking, cross-fit trên OOF, cùng nguyên tắc mục 16 baseline) — quán quân `CHAMPION_COMBINER` được CHỌN và KHOÁ ở đây dựa trên AUPRC OOF, TRƯỚC khi chạm TEST. Ensemble chỉ thắng nếu thực sự tốt hơn từng kiến trúc đơn lẻ trên OOF.


In [ ]:
from sklearn.linear_model import LogisticRegression


def _logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=np.float64), eps, 1 - eps)
    return np.log(p / (1 - p))


def crossfit_platt(p_raw, y_true, fold_id):
    """Fold k được hiệu chuẩn bằng bộ Platt fit trên OOF của các fold KHÁC -- ước lượng TRUNG
    THỰC (không tự chấm điểm mình)."""
    out = np.full(len(p_raw), np.nan)
    for k in np.unique(fold_id):
        fit_mask = fold_id != k
        apply_mask = fold_id == k
        lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
        lr.fit(_logit(p_raw[fit_mask]).reshape(-1, 1), y_true[fit_mask].astype(int))
        out[apply_mask] = lr.predict_proba(_logit(p_raw[apply_mask]).reshape(-1, 1))[:, 1]
    return out


oof_y_true = pool_df['OMI'].to_numpy().astype(float)
_fold_id_arr = pool_df["_fold"].to_numpy()

# =============================================================================
# Tier 2: train K-fold cho TỪNG kiến trúc trong BACKBONE_ARCH_LIST -- resnet1d dùng lại run_tag
# _tier1 (đã train ở Tier 1, chỉ eval lại), inceptiontime1d train MỚI với run_tag _tier2ic, luôn
# full fine-tune (không qua screening, xem unfreeze_all_backbone ở Bước 5).
# =============================================================================
oof_y_prob_by_arch = {}
oof_y_prob_cal_by_arch = {}
PLATT_FULL_by_arch = {}

for arch in BACKBONE_ARCH_LIST:
    print(f"\n{'=' * 20} Kiến trúc: {arch} {'=' * 20}")
    _suffix = RUN_TAG_SUFFIX_BY_ARCH[arch]
    _level = best_level if arch == "resnet1d" else None
    _level_name = FREEZE_LEVELS[_level] if _level is not None else "full_finetune (Tier 2, không qua screening)"
    _oof_y_prob = np.full(len(pool_df), np.nan)
    for fold_id in sorted(pool_df["_fold"].unique()):
        print(f"\n=== [{arch}] Fold {fold_id}/{pool_df['_fold'].nunique() - 1} (mức: {_level_name}) ===")
        tr_df = pool_df[pool_df["_fold"] != fold_id]
        va_df = pool_df[pool_df["_fold"] == fold_id]
        tr_loader = DataLoader(ECGBinaryDataset(tr_df, pool_cache, augment=True), batch_size=BATCH_SIZE,
                                shuffle=True, drop_last=True)
        va_loader = DataLoader(ECGBinaryDataset(va_df, pool_cache), batch_size=BATCH_SIZE, shuffle=False)
        assert len(tr_loader) > 0, f"[{arch}] Fold {fold_id}: train_loader rỗng"
        res = train_one_run(tr_loader, va_loader, _level, run_tag=f"{TASK}_fold{fold_id}{_suffix}",
                             n_epochs=2 if RUN_MODE == "debug" else 30, arch=arch)
        _oof_y_prob[va_df.index.to_numpy()] = res["oof_y_prob"]
    assert not np.isnan(_oof_y_prob).any(), f"[{arch}] Có record chưa được gán OOF -- kiểm tra lại K-fold"

    _oof_y_prob_cal = crossfit_platt(_oof_y_prob, oof_y_true, _fold_id_arr)
    _platt_full = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
    _platt_full.fit(_logit(_oof_y_prob).reshape(-1, 1), oof_y_true.astype(int))

    oof_y_prob_by_arch[arch] = _oof_y_prob
    oof_y_prob_cal_by_arch[arch] = _oof_y_prob_cal
    PLATT_FULL_by_arch[arch] = _platt_full
    print(f"[{arch}] OOF AUPRC -- thô: {average_precision_score(oof_y_true, _oof_y_prob):.4f} | "
          f"sau hiệu chuẩn: {average_precision_score(oof_y_true, _oof_y_prob_cal):.4f}")

# Biến tương thích ngược (dùng ở Bước 8 chỉ cho mục đích chẩn đoán "không hiệu chuẩn thì sao").
oof_y_prob = oof_y_prob_by_arch["resnet1d"]


def apply_platt(p_raw, arch):
    return PLATT_FULL_by_arch[arch].predict_proba(_logit(np.asarray(p_raw)).reshape(-1, 1))[:, 1]


# =============================================================================
# Vườn ensemble (Tier 2) -- tinh thần mục 16 stemi_pipeline_optimized_v3.ipynb: vài cách tổ hợp
# được CHẤM ĐIỂM cross-fit CHỈ trên OOF của POOL, quán quân KHOÁ tại đây trước khi chạm TEST. Ở
# đây "candidate" là {len(BACKBONE_ARCH_LIST)} kiến trúc (resnet1d Tier 1, inceptiontime1d Tier 2)
# thay vì 8-10 kiến trúc như baseline -- ít lựa chọn hơn nhưng cùng nguyên tắc. Từng kiến trúc
# đơn lẻ cũng là ứng viên -- ensemble chỉ thắng nếu THỰC SỰ tốt hơn trên OOF, không mặc định chọn.
# =============================================================================
_archs = BACKBONE_ARCH_LIST
_M_cal = np.column_stack([oof_y_prob_cal_by_arch[a] for a in _archs])
_M_logit = _logit(_M_cal)


def _comb_mean(M):
    return M.mean(axis=1)


def _comb_logit_mean(M_logit):
    return 1.0 / (1.0 + np.exp(-M_logit.mean(axis=1)))


def _crossfit_stack(M_logit, y_true, fold_id):
    out = np.full(len(y_true), np.nan)
    for k in np.unique(fold_id):
        fit_mask = fold_id != k
        apply_mask = fold_id == k
        lr = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000)
        lr.fit(M_logit[fit_mask], y_true[fit_mask].astype(int))
        out[apply_mask] = lr.predict_proba(M_logit[apply_mask])[:, 1]
    return out


_ensemble_candidates = {
    "mean": _comb_mean(_M_cal),
    "logit_mean": _comb_logit_mean(_M_logit),
    "stacking": _crossfit_stack(_M_logit, oof_y_true, _fold_id_arr),
}
for a in _archs:
    _ensemble_candidates[f"single_{a}"] = oof_y_prob_cal_by_arch[a]

_ensemble_scores = {name: average_precision_score(oof_y_true, p) for name, p in _ensemble_candidates.items()}
print("\nĐiểm OOF AUPRC (cross-fit, không nhìn TEST) của các cách tổ hợp:")
for _name, _score in sorted(_ensemble_scores.items(), key=lambda kv: -kv[1]):
    print(f"  {_name:<20} {_score:.4f}")

CHAMPION_COMBINER = max(_ensemble_scores, key=_ensemble_scores.get)
oof_y_prob_cal = _ensemble_candidates[CHAMPION_COMBINER]  # dùng cho Bước 8 (tương thích tên biến cũ)
print(f"\nQuán quân (khoá trước khi chạm TEST): {CHAMPION_COMBINER}")

if CHAMPION_COMBINER == "stacking":
    STACKER_FULL = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000)
    STACKER_FULL.fit(_M_logit, oof_y_true.astype(int))

pool_df["_oof_prob_cal"] = oof_y_prob_cal
for a in _archs:
    pool_df[f"_oof_prob_{a}"] = oof_y_prob_by_arch[a]
pool_df[["record_id", 'OMI', "_fold", "_oof_prob_cal"] + [f"_oof_prob_{a}" for a in _archs]].to_csv(
    FINETUNE_OOF_DIR / f"{TASK}_pool_oof.csv", index=False)


## Bước 8 — Khoá threshold trên POOL-OOF (Youden's J + biên an toàn)

Lấy Sensitivity mà điểm cân bằng Youden's J đạt trên POOL-OOF làm mục tiêu cần bảo vệ, rồi áp
**biên an toàn `SENS_MARGIN_Z`** giống hệt STEMI: đo SD của Sensitivity giữa 5 fold tại ngưỡng
đó, nhắm cao hơn `z=1.28` lần SD trước khi khoá — chống lỗi "khoá đúng % trên POOL nhưng tụt
hẳn trên TEST" mà lần chạy trước gặp (Sensitivity POOL 75.5% → chỉ còn 56.3% trên TEST).

**Cập nhật:** khoá ngưỡng trên `oof_y_prob_cal` (đã hiệu chuẩn Platt ở Bước 7) thay vì xác suất
thô -- cell in thêm dòng chẩn đoán so sánh SD Sensitivity CÓ/KHÔNG hiệu chuẩn để thấy rõ mức sửa.

In [ ]:
def threshold_at_sensitivity(y_true, y_prob, target_sens):
    """Ngưỡng LỚN NHẤT sao cho Sensitivity thực tế vẫn >= target_sens -- dò trực tiếp trên điểm
    số của các ca dương (không nội suy tuyến tính trên ROC dạng bậc thang, vốn có thể cho kết
    quả tụt dưới mục tiêu). Verbatim stemi_pipeline_optimized_v3.ipynb mục 19."""
    y_true = np.asarray(y_true).astype(int)
    pos_scores = np.sort(np.asarray(y_prob)[y_true == 1])[::-1]
    n_pos = len(pos_scores)
    k = min(int(np.ceil(target_sens * n_pos)), n_pos)
    return float(pos_scores[k - 1]) if k > 0 else 1.0

def lock_threshold_youden(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j = tpr - fpr
    return float(thresholds[np.argmax(j)])

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "sensitivity": tp / (tp + fn) if (tp + fn) else float("nan"),
        "specificity": tn / (tn + fp) if (tn + fp) else float("nan"),
        "ppv": tp / (tp + fp) if (tp + fp) else float("nan"),
        "npv": tn / (tn + fn) if (tn + fn) else float("nan"),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }

# =============================================================================
# Chốt ngưỡng CÓ BIÊN AN TOÀN -- verbatim cơ chế mục 19 stemi_pipeline_optimized_v3.ipynb.
#
# Vấn đề đang sửa: ngưỡng chốt trên POOL-OOF để đạt Sensitivity = S gần như KHÔNG BAO GIỜ cho
# đúng S trên TEST -- TEST chỉ có vài trăm ca dương nên Sensitivity ở đó dao động thuần do cỡ
# mẫu. Lần chạy trước của chính notebook này rơi đúng vào chiều xấu (xem phân tích kết quả):
# khoá đúng mục tiêu trên POOL nhưng Sensitivity trên TEST tụt hàng chục điểm % -- vi phạm
# chính sách lâm sàng đặt ra. stemi_pipeline_optimized_v3.ipynb từng dính lỗi y hệt ở v2
# (chốt 91% trên POOL -> chỉ đạt 88,9% trên TEST) và đã vá bằng cơ chế dưới đây.
#
# Cách sửa (thuần POOL, không nhìn TEST): tại đúng ngưỡng ứng với mục tiêu, đo Sensitivity đạt
# được trong TỪNG fold rồi lấy SD giữa các fold (mỗi fold cỡ tương đương TEST, nên SD này ước
# lượng đúng mức dao động sẽ gặp khi chuyển sang TEST). Nhắm cao hơn mục tiêu z=1.28 lần SD đó
# để xác suất đạt mục tiêu thật trên TEST ~90%. Trần SENS_MARGIN_MAX_TARGET chặn trường hợp SD
# lớn bất thường đẩy mục tiêu sát 1.0 (ngưỡng tụt xuống dự đoán MỌI ca dương, Specificity=0 --
# hỏng âm thầm mà vẫn in kết quả bình thường).
# =============================================================================

SENS_MARGIN_Z = 1.28            # verbatim stemi_pipeline_optimized_v3.ipynb
SENS_MARGIN_MAX_TARGET = 0.97   # trần bắt buộc

def fold_sensitivity_sd(y_true, y_prob, fold_id, thr):
    """SD của Sensitivity giữa các fold tại CÙNG một ngưỡng."""
    sens = []
    for k in np.unique(fold_id):
        m = fold_id == k
        yk, pk = y_true[m], y_prob[m]
        if yk.sum() == 0:
            continue
        sens.append(float((pk[yk == 1] >= thr).mean()))
    return float(np.std(sens, ddof=1)) if len(sens) > 1 else 0.0

def threshold_with_margin(y_true, y_prob, fold_id, target, z=SENS_MARGIN_Z):
    t_plain = threshold_at_sensitivity(y_true, y_prob, target)
    sd = fold_sensitivity_sd(y_true, y_prob, fold_id, t_plain)
    raw_adj = target + z * sd
    target_adj = float(min(raw_adj, SENS_MARGIN_MAX_TARGET))
    capped = raw_adj > SENS_MARGIN_MAX_TARGET + 1e-12
    return threshold_at_sensitivity(y_true, y_prob, target_adj), target_adj, sd, capped

if THRESHOLD_POLICY == "sensitivity_floor":
    _base_target = SENSITIVITY_FLOOR
elif THRESHOLD_POLICY == "youden":
    # Youden chỉ cho 1 điểm cân bằng, không có "sàn" để bảo vệ như sensitivity_floor -- dùng
    # chính Sensitivity mà điểm cân bằng đó đạt trên POOL-OOF (đã hiệu chuẩn) làm mục tiêu cần
    # bảo vệ khi chuyển sang TEST, rồi áp CÙNG cơ chế biên an toàn bên trên.
    _youden_thr = lock_threshold_youden(oof_y_true, oof_y_prob_cal)
    _base_target = compute_metrics(oof_y_true, oof_y_prob_cal, _youden_thr)["sensitivity"]
else:
    raise ValueError(f"THRESHOLD_POLICY không hợp lệ: {THRESHOLD_POLICY}")

# Chẩn đoán: SD Sensitivity của resnet1d ĐƠN LẺ, KHÔNG hiệu chuẩn, KHÔNG ensemble -- để
# thấy rõ mức cải thiện qua từng lớp (calibration + ensemble) so với Tier 1 thô.
_diag_thr, _diag_target_adj, _diag_sd, _diag_capped = threshold_with_margin(
    oof_y_true, oof_y_prob, pool_df["_fold"].to_numpy(), _base_target)
print(f"(Chẩn đoán, không dùng) SD Sensitivity trên resnet1d thô (chưa hiệu chuẩn, chưa ensemble): "
      f"{_diag_sd:.4f} -> nếu dùng nguyên trạng thái này, mục tiêu sẽ bị nâng lên {_diag_target_adj:.4f}")

LOCKED_THRESHOLD, _target_adj, _fold_sd, _capped = threshold_with_margin(
    oof_y_true, oof_y_prob_cal, pool_df["_fold"].to_numpy(), _base_target)

_oof_metrics = compute_metrics(oof_y_true, oof_y_prob_cal, LOCKED_THRESHOLD)
print(f"Threshold khoá trên POOL-OOF đã hiệu chuẩn ({THRESHOLD_POLICY}, biên an toàn z={SENS_MARGIN_Z}): "
      f"{LOCKED_THRESHOLD:.4f}")
print(f"  Mục tiêu gốc: {_base_target:.4f} | SD Sensitivity giữa 5 fold (đã hiệu chuẩn): {_fold_sd:.4f} | "
      f"mục tiêu đã nâng: {_target_adj:.4f}" + (" -- ĐÃ CHẠM TRẦN AN TOÀN" if _capped else ""))
print(json.dumps(_oof_metrics, indent=2))


## Bước 9 — Tier 2: FINAL + bagging cho mọi kiến trúc, tổ hợp bằng quán quân đã khoá

Với TỪNG kiến trúc: train FINAL (retrain trên toàn POOL) + bagging (trung bình 5 checkpoint fold từ Bước 7, đã hiệu chuẩn Platt). Sau đó áp ĐÚNG `CHAMPION_COMBINER` đã khoá ở Bước 7 lên các vector bagging TEST của từng kiến trúc để ra `test_y_prob` cuối cùng — không chọn lại cách tổ hợp ở đây (tránh rò rỉ TEST vào lựa chọn).


In [ ]:
test_loader = DataLoader(ECGBinaryDataset(test_df, test_cache), batch_size=BATCH_SIZE, shuffle=False)


def predict_on_loader(ckpt_path, loader, arch="resnet1d"):
    m = FinetuneModel(arch=arch).to(DEVICE)
    # weights_only=False: checkpoint tự tạo trong chính notebook này (chứa val_auprc kiểu numpy
    # scalar) -- mặc định weights_only=True của torch>=2.6 sẽ raise UnpicklingError.
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    y_true_, y_prob_ = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y_prob_.append(torch.sigmoid(m(x)).cpu().numpy())
            y_true_.append(y.numpy())
    return np.concatenate(y_true_), np.concatenate(y_prob_)


# --- Bagging: trung bình xác suất của 5 checkpoint fold ĐÃ CÓ SẴN từ Bước 7 (không train thêm)
# -- stemi_pipeline_optimized_v3.ipynb mặc định PRIMARY_PREDICTOR="bagging" vì đo được
# +0.03..+0.05 AUPRC so với 1 model FINAL retrain trên toàn POOL. Quyết định CỐ ĐỊNH trước khi
# nhìn TEST (không chọn theo điểm TEST, tránh rò rỉ lựa chọn model vào TEST).
PRIMARY_PREDICTOR = "bagging"
bag_y_prob_by_arch = {}
test_y_true = None

for arch in BACKBONE_ARCH_LIST:
    print(f"\n{'=' * 20} Kiến trúc: {arch} {'=' * 20}")
    _suffix = RUN_TAG_SUFFIX_BY_ARCH[arch]
    _level = best_level if arch == "resnet1d" else None
    full_pool_loader = DataLoader(ECGBinaryDataset(pool_df, pool_cache, augment=True), batch_size=BATCH_SIZE,
                                  shuffle=True, drop_last=True)
    # screen_val_loader: theo dõi trong lúc train FINAL (không dùng để chọn gì trên TEST).
    final_res = train_one_run(full_pool_loader, screen_val_loader, _level,
                              run_tag=f"{TASK}_FINAL{_suffix}", n_epochs=2 if RUN_MODE == "debug" else 30,
                              arch=arch)

    _yt, _final_y_prob_raw = predict_on_loader(final_res["best_ckpt"], test_loader, arch=arch)
    if test_y_true is None:
        test_y_true = _yt
    else:
        assert np.array_equal(_yt, test_y_true), f"[{arch}] Thứ tự TEST lệch với kiến trúc khác"
    # Áp CÙNG bộ hiệu chuẩn Platt (fit trên OOF ở Bước 7) lên xác suất thô trên TEST -- threshold
    # đã khoá ở Bước 8 nằm trên thang ĐÃ hiệu chuẩn.
    final_y_prob = apply_platt(_final_y_prob_raw, arch)

    _fold_ids = sorted(pool_df["_fold"].unique())
    _bag_probs = []
    for fold_id in _fold_ids:
        _ckpt_path = FINETUNE_MODELS_DIR / f"{TASK}_fold{fold_id}{_suffix}_best.pt"
        _yt2, _yp_raw = predict_on_loader(_ckpt_path, test_loader, arch=arch)
        assert np.array_equal(_yt2, test_y_true), f"[{arch}] Thứ tự TEST lệch giữa fold {fold_id} và FINAL"
        _bag_probs.append(apply_platt(_yp_raw, arch))
    bag_y_prob = np.mean(_bag_probs, axis=0)
    bag_y_prob_by_arch[arch] = bag_y_prob

    _auprc_final = average_precision_score(test_y_true, final_y_prob)
    _auprc_bag = average_precision_score(test_y_true, bag_y_prob)
    print(f"[{arch}] (Tham khảo, KHÔNG dùng để quyết định) FINAL AUPRC={_auprc_final:.4f} | "
          f"Bagging ({len(_fold_ids)} fold) AUPRC={_auprc_bag:.4f}")

print(f"\nPredictor chính dùng cho từng kiến trúc: {PRIMARY_PREDICTOR.upper()}")

# --- Tổ hợp bằng ĐÚNG quán quân đã khoá ở Bước 7 (CHAMPION_COMBINER) -- không chọn lại ở đây,
# tránh rò rỉ TEST vào lựa chọn cách tổ hợp. ---
_archs = BACKBONE_ARCH_LIST
_M_test = np.column_stack([bag_y_prob_by_arch[a] for a in _archs])
if CHAMPION_COMBINER == "mean":
    test_y_prob = _M_test.mean(axis=1)
elif CHAMPION_COMBINER == "logit_mean":
    test_y_prob = 1.0 / (1.0 + np.exp(-_logit(_M_test).mean(axis=1)))
elif CHAMPION_COMBINER == "stacking":
    test_y_prob = STACKER_FULL.predict_proba(_logit(_M_test))[:, 1]
elif CHAMPION_COMBINER.startswith("single_"):
    test_y_prob = bag_y_prob_by_arch[CHAMPION_COMBINER.replace("single_", "")]
else:
    raise ValueError(f"CHAMPION_COMBINER không hợp lệ: {CHAMPION_COMBINER}")

test_metrics = compute_metrics(test_y_true, test_y_prob, LOCKED_THRESHOLD)
test_auprc = average_precision_score(test_y_true, test_y_prob)
test_auroc = roc_auc_score(test_y_true, test_y_prob) if len(set(test_y_true)) > 1 else float("nan")
print(f"\nTEST (quán quân={CHAMPION_COMBINER}, {PRIMARY_PREDICTOR}) -- AUPRC={test_auprc:.4f} AUROC={test_auroc:.4f}")
print(json.dumps(test_metrics, indent=2))

_out_df = pd.DataFrame({"record_id": test_df["record_id"], 'OMI': test_y_true, "y_prob": test_y_prob})
for a in _archs:
    _out_df[f"y_prob_bag_{a}"] = bag_y_prob_by_arch[a]
_out_df.to_csv(FINETUNE_OOF_DIR / f"{TASK}_test_predictions.csv", index=False)


## Bước 10 — So sánh với baseline: bootstrap (1-model) + điểm trực tiếp với báo cáo

`stemi_pipeline_optimized_v3.ipynb` (mục 22) lưu xác suất dự đoán TEST của **từng model đơn** (`bag_{model}`/`fin_{model}`) — nhưng model chính (champion) của báo cáo luôn là một ensemble 8 kiến trúc có trọng số fit lúc chạy, không lưu ra file. Cell dưới vẫn giữ so sánh bootstrap `bag_ResNet1D` (tách bạch hiệu ứng pretrain khỏi hiệu ứng ensemble) và **thêm** so điểm trực tiếp `test_metrics` (ensemble Tier 2 ở đây) với đúng các con số trong báo cáo (`REPORT_BASELINE`) — đây là phép so cuối cùng quyết định có đạt mục tiêu (FN và FP đều thấp hơn báo cáo) hay chưa.


In [ ]:
def bootstrap_delta_auprc(y_true, y_prob_a, y_prob_b, n_boot=2000, seed=SEED):
    """A=pretrain-init (model ở đây), B=baseline from-scratch. Trả về delta điểm ước lượng, CI 95%,
    p-value xấp xỉ, và có ý nghĩa thống kê hay không (CI không chứa 0)."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    deltas = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        if yt.sum() == 0 or yt.sum() == n:
            idx = rng.integers(0, n, size=n)
            yt = y_true[idx]
        deltas[b] = (average_precision_score(yt, y_prob_a[idx])
                     - average_precision_score(yt, y_prob_b[idx]))
    point = average_precision_score(y_true, y_prob_a) - average_precision_score(y_true, y_prob_b)
    ci_lo, ci_hi = np.percentile(deltas, [2.5, 97.5])
    p_value = 2 * min((deltas <= 0).mean(), (deltas >= 0).mean())
    return {"point_delta_auprc": point, "ci95_lo": ci_lo, "ci95_hi": ci_hi,
            "p_value_approx": p_value, "significant": not (ci_lo <= 0 <= ci_hi)}


def find_baseline_predictions_npz():
    """Tìm file .npz xác suất TEST đã lưu bởi omi_pipeline_optimized_v3.ipynb (mục 22).
    Không có quyền chọn cột thay bạn (champion là ensemble, không nằm trong file) -- chỉ giúp
    tìm file + liệt kê cột sẵn có."""
    results_dir = DATA_DRIVE_PROJECT / "outputs" / "results"
    if not results_dir.exists():
        print(f"Không thấy {results_dir} -- chạy omi_pipeline_optimized_v3.ipynb (mục 22) trước, "
              f"hoặc điền thủ công đường dẫn npz vào biến bên dưới.")
        return None
    candidates = sorted(results_dir.glob("omi_test_predictions_*.npz"))
    if not candidates:
        print(f"Không thấy omi_test_predictions_*.npz trong {results_dir}.")
        return None
    return candidates[-1]


def load_baseline_column(npz_path, column, test_df=test_df):
    """Nạp 1 cột xác suất từ .npz baseline, sắp lại đúng thứ tự record_id của test_df ở đây --
    thứ tự TEST giữa 2 notebook có thể khác nhau dù cùng tập bệnh nhân."""
    data = np.load(npz_path, allow_pickle=True)
    by_record = dict(zip(data["record_stem"].astype(str), data[column]))
    missing = [r for r in test_df["record_id"] if r not in by_record]
    assert not missing, f"{len(missing)} record_id của TEST ở đây không có trong npz baseline: {missing[:5]}..."
    return np.array([by_record[r] for r in test_df["record_id"]])


_baseline_npz = find_baseline_predictions_npz()
baseline_y_prob = None
_baseline_col = "bag_ResNet1D"   # cùng kiến trúc backbone với pretrain-init (ResNet1D), cùng
                                    # predictor bagging -- so sánh 1-model-vs-1-model, tách bạch
                                    # hiệu ứng pretrain khỏi hiệu ứng ensemble 8 kiến trúc.
if _baseline_npz is not None:
    _keys = [k for k in np.load(_baseline_npz).files if k.startswith(("bag_", "fin_"))]
    print(f"Tìm thấy: {_baseline_npz}")
    if _baseline_col in _keys:
        baseline_y_prob = load_baseline_column(_baseline_npz, _baseline_col)
        print(f"Dùng cột '{_baseline_col}' (cùng kiến trúc, cùng predictor bagging, from-scratch) "
              f"để so sánh công bằng -- tách bạch hiệu ứng pretrain khỏi hiệu ứng ensemble 8 model.")
    else:
        print(f"Không thấy cột '{_baseline_col}' trong npz -- cột có sẵn ({len(_keys)}): {_keys}")
        print('  baseline_y_prob = load_baseline_column(_baseline_npz, "bag_<tên_model>")')

if baseline_y_prob is not None:
    comparison = bootstrap_delta_auprc(test_y_true, test_y_prob, np.asarray(baseline_y_prob))
    print(json.dumps({k: (float(v) if hasattr(v, "item") else v) for k, v in comparison.items()},
                     indent=2))
    if comparison["significant"] and comparison["point_delta_auprc"] > 0:
        print("\n>>> Pretrain-init CẢI THIỆN có ý nghĩa thống kê so với baseline -- cân nhắc đưa vào FINAL/ensemble.")
    elif comparison["significant"] and comparison["point_delta_auprc"] < 0:
        print("\n>>> Pretrain-init KÉM HƠN baseline có ý nghĩa thống kê -- giữ nguyên from-scratch.")
    else:
        print("\n>>> Chênh lệch không có ý nghĩa thống kê -- không đủ bằng chứng để đổi.")
else:
    print("Điền baseline_y_prob trước khi kết luận -- xem gợi ý ở trên (tuỳ chọn, không chặn chạy full).")

# =============================================================================
# So SÁNH TRỰC TIẾP với báo cáo (Bao_cao_nghien_cuu.docx phần II) -- ensemble 8 kiến trúc
# from-scratch, mục tiêu cần vượt qua. Không có bootstrap CI ở đây vì không có xác suất TEST gốc
# của "champion" ensemble 8 model đó (chỉ có confusion-matrix cuối cùng), nên so điểm trực tiếp.
# =============================================================================
REPORT_BASELINE = {"sensitivity": 0.8678, "specificity": 0.8149, "ppv": 0.2443, "npv": 0.9889, "tp": 151, "fn": 23, "fp": 467, "tn": 2055}
print("\n" + "=" * 70)
print(f"SO VỚI BÁO CÁO (ensemble 8 kiến trúc from-scratch, mục tiêu cần vượt) -- TASK={TASK}")
print("=" * 70)
for _k in ["sensitivity", "specificity", "ppv", "npv"]:
    print(f"  {_k:<12} hiện tại={test_metrics[_k]:.4f}  báo cáo={REPORT_BASELINE[_k]:.4f}  "
          f"delta={test_metrics[_k] - REPORT_BASELINE[_k]:+.4f}")
print(f"  TP/FN/FP/TN  hiện tại={test_metrics['tp']}/{test_metrics['fn']}/{test_metrics['fp']}/{test_metrics['tn']}  "
      f"báo cáo={REPORT_BASELINE['tp']}/{REPORT_BASELINE['fn']}/{REPORT_BASELINE['fp']}/{REPORT_BASELINE['tn']}")
_beats_report = (test_metrics["fn"] <= REPORT_BASELINE["fn"] and test_metrics["fp"] <= REPORT_BASELINE["fp"])
print("\n>>> " + ("ĐẠT mục tiêu: FN và FP đều <= báo cáo." if _beats_report else
                   "CHƯA đạt mục tiêu: còn ít nhất 1 trong FN/FP cao hơn báo cáo."))


In [ ]:
def export_run_manifest():
    manifest = {
        "task": TASK,
        "run_mode": RUN_MODE,
        "created_at": datetime.utcnow().isoformat(),
        "source_backbone_manifest": str(_manifest_path),
        "source_backbone_sha256": _expected_sha256,
        "freeze_level_screening": {FREEZE_LEVELS[k]: v for k, v in screening_results.items()},
        "best_freeze_level": FREEZE_LEVELS[best_level],
        "threshold_policy": THRESHOLD_POLICY,
        "locked_threshold": LOCKED_THRESHOLD,
        "pool_oof_metrics": _oof_metrics,
        "test_metrics": test_metrics,
        "test_auprc": test_auprc,
        "test_auroc": test_auroc,
        "backbone_archs": BACKBONE_ARCH_LIST,
        "champion_combiner": CHAMPION_COMBINER,
        "ensemble_oof_scores": {k: float(v) for k, v in _ensemble_scores.items()},
        "n_pool": len(pool_df), "n_test": len(test_df),
    }
    path = FINETUNE_OOF_DIR / f"{TASK}_finetune_run_manifest.json"
    path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    print(f"Manifest: {path}")
    return path

_ = export_run_manifest()


## Kết thúc

Kết quả (mức freeze/unfreeze thắng, threshold khoá, metric POOL-OOF/TEST, so sánh bootstrap với
baseline) đã lưu tại `manifests/omi_finetune_oof/omi_finetune_run_manifest.json`.

**Sẵn sàng chạy `RUN_MODE="full"`** ngay khi notebook 14 chạy xong (`RUN_MODE="full"`, đã sinh
`*_run_manifest.json` + `*_backbone_only.pt`) — Bước 2 đã đọc dữ liệu ACS-ECG 2026 thật và tái lập
đúng POOL/TEST/fold của `omi_pipeline_optimized_v3.ipynb`.

**Việc còn lại chỉ có tính tham khảo, không chặn chạy full:**
- **Bước 10** — `baseline_y_prob`: cell đã tự tìm `omi_test_predictions_*.npz` và liệt kê các
  cột model đơn có sẵn; điền 1 dòng để so nhanh, hoặc theo hướng dẫn ở markdown Bước 10 nếu muốn
  so đúng với ensemble champion của baseline.

Đưa kết quả cuối cùng vào `Bao_cao_nghien_cuu.docx` (mục 2.2 hoặc 2.3 tương ứng OMI), diễn giải
theo đúng khung: đây là bước trong quy trình phát triển mô hình (giống "Fine-tune foundation model"
ở Phần I), không phải external validation.